In [ ]:
!pip install lxml_html_clean
!pip install newspaper3k

  Using cached lxml_html_clean-0.4.4-py3-none-any.whl.metadata (2.4 kB)
  Using cached lxml-6.0.2-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (3.6 kB)
Using cached lxml_html_clean-0.4.4-py3-none-any.whl (14 kB)
Using cached lxml-6.0.2-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (5.3 MB)


In [ ]:
import requests
from newspaper import Article, Config
import json
from datetime import datetime
import uuid
import time
import re
import random
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

TICKERS = [
    {"symbol": "BTCUSDT", "name": "Bitcoin", "keywords": ["bitcoin", "btc"]},
    {"symbol": "ETHUSDT", "name": "Ethereum", "keywords": ["ethereum", "eth"]},
    {"symbol": "BNBUSDT", "name": "BNB", "keywords": ["bnb", "binance coin", "bsc"]},
    {"symbol": "LTCUSDT", "name": "Litecoin", "keywords": ["litecoin", "ltc"]},
    {"symbol": "XRPUSDT", "name": "XRP", "keywords": ["ripple", "xrp"]},
]

BASE_SITEMAP = "https://cointelegraph.com/sitemap/post-{}.xml"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

DELAY_RANGE = (0.5, 1.5)  # giảm delay cho nhanh hơn
MAX_WORKERS = 5           # số luồng

START_YEAR = 2017
END_YEAR = 2026



# ** Xin chú ý: Mỗi người có start_page và end_page khác nhau
# Tránh chạy nhầm lên data người khác đã crawl trước
START_PAGE = 1       
END_PAGE = 10

seen_urls = set()


def generate_id():
    return str(uuid.uuid4()).replace('-', '')[:24]


def get_publish_date(article, html_content=None):
    try:
        soup = BeautifulSoup(html_content or article.html, 'html.parser')

        # JSON-LD
        json_ld = soup.find_all('script', type='application/ld+json')
        for script in json_ld:
            try:
                data = json.loads(script.string)
                items = data if isinstance(data, list) else [data]
                for item in items:
                    date_str = item.get('datePublished') or item.get('uploadDate')
                    if date_str:
                        return datetime.fromisoformat(date_str.replace('Z', '+00:00'))
            except:
                continue

        # fallback
        selectors = [
            {'name': 'meta', 'attrs': {'property': 'article:published_time'}},
            {'name': 'meta', 'attrs': {'name': 'publish-date'}},
            {'name': 'time', 'attrs': {'datetime': True}}
        ]

        for s in selectors:
            elem = soup.find(s['name'], s['attrs'])
            if elem:
                date_str = elem.get('content') or elem.get('datetime')
                if date_str:
                    return datetime.fromisoformat(date_str.replace('Z', '+00:00'))

    except:
        pass

    return article.publish_date


def scrape_article_data(url):
    try:
        config = Config()
        config.browser_user_agent = HEADERS['User-Agent']
        config.request_timeout = 20
        config.memoize_articles = False

        article = Article(url, config=config)
        article.download()
        article.parse()

        dt = get_publish_date(article, article.html)

        if dt:
            formatted_date = dt.strftime("%d-%m-%Y %I:%M %p")
            year = dt.year
        else:
            formatted_date = "Khong xac dinh ngay dang"
            year = None

        return {
            "title": article.title.strip() if article.title else "Khong tieu de",
            "text": article.text.strip() if len(article.text) > 50 else "Noi dung qua ngan",
            "date": formatted_date,
            "year": year
        }

    except Exception as e:
        print(f"Loi scrape: {url} | {e}")
        return None


def process_url(url, current_page):
    if url in seen_urls:
        return None

    # match ticker
    matched_tickers = []
    for ticker in TICKERS:
        if any(kw in url.lower() for kw in ticker['keywords']):
            matched_tickers.append(ticker)

    if not matched_tickers:
        return None

    # scrape
    info = scrape_article_data(url)
    if not info:
        return None

    if info['year'] and (info['year'] < START_YEAR or info['year'] > END_YEAR):
        return None

    seen_urls.add(url)

    now = datetime.now()

    return {
        "_id": generate_id(),
        "metadata": {
            "Date": now.strftime("%Y-%m-%d"),
            "Time": now.strftime("%H:%M:%S")
        },
        "link": url,
        "post date": info['date'],
        "summary": info['title'],
        "context": info['text'],
        "ticket symbol": ", ".join([t['symbol'] for t in matched_tickers]),
        "ticket name": ", ".join([t['name'] for t in matched_tickers]),
        "keyword": ", ".join([kw for t in matched_tickers for kw in t['keywords']]),
        "page": current_page
    }


def crawl_all_coins_single_file():
    output_data = []
    collected = 0

    try:
        for current_page in range(START_PAGE, END_PAGE + 1):
            print(f"\n=== Dang quet Sitemap {current_page} ===", flush=True)

            # retry sitemap
            success = False
            for attempt in range(3):
                resp = requests.get(BASE_SITEMAP.format(current_page), headers=HEADERS, timeout=30)
                if resp.status_code == 200:
                    success = True
                    break
                print(f"Bi chan {resp.status_code}, retry {attempt+1}")
                time.sleep(10)

            if not success:
                print("Bo qua sitemap nay")
                continue

            urls = re.findall(r'<loc>(https?://cointelegraph\.com/[^<]+)</loc>', resp.text)

            # multi-thread
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [executor.submit(process_url, url, current_page) for url in urls]

                for future in as_completed(futures):
                    result = future.result()

                    if result:
                        output_data.append(result)
                        collected += 1

                        # print realtime
                        print(f"\n[{collected}] {result['link']}", flush=True)
                        print(json.dumps(result, ensure_ascii=False, indent=2))

                        # save temp mỗi 10 bài
                        if collected % 10 == 0:
                            with open("temp.json", "w", encoding="utf-8") as f:
                                json.dump(output_data, f, ensure_ascii=False, indent=2)
                            print(">>> Da luu tam 10 bai")

                        time.sleep(random.uniform(*DELAY_RANGE))

    except KeyboardInterrupt:
        print("Dung thu cong, dang luu file...")

    finally:
        if output_data:
            filename = f"cointelegraph_{datetime.now().strftime('%H%M')}.json"
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(output_data, f, ensure_ascii=False, indent=2)

            print(f"\nHoan thanh: {collected} bai -> {filename}")
        else:
            print("Khong co du lieu")


if __name__ == "__main__":
    crawl_all_coins_single_file()